In [ ]:
# 1. Install the package in editable mode from the local project root
!pip install -e ".[all]"
!pip install reportlab

# 2. Optional: Download spaCy model for Presidio PII
!python -m spacy download en_core_web_sm

In [ ]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

# ---------------------------------------------------------------------------
# Example LLM configuration
# ---------------------------------------------------------------------------
try:
    groq_api_key = os.getenv("GROQ_API_KEY")
except Exception:
    groq_api_key = None

groq_config = {
    "base_url": "https://api.example.com/v1",
    "api_key": groq_api_key,
    "model_name": "example-model",
    "temperature": 0.0
}

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def generate_test_pdf(filename: str = "sample_doc.pdf"):
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    # Title
    story.append(Paragraph("Sample Document Overview", styles["Heading1"]))
    story.append(Spacer(1, 12))

    # Example text
    p1 = (
        "Example Organization is evaluating a sample procurement notice. "
        "Primary contact: Example Person, Email: user@example.com, "
        "Phone: +1-555-0100, SSN: 123-45-6789, Host IP: 192.168.1.100."
    )
    story.append(Paragraph(p1, styles["Normal"]))
    story.append(Spacer(1, 16))

    # Table Section
    story.append(Paragraph("Sample Expenditures", styles["Heading2"]))
    story.append(Spacer(1, 8))
    table_data = [
        ["Item", "Amount", "Currency"],
        ["Hosting", "250.00", "USD"],
        ["Software", "79.99", "USD"],
        ["Support", "1200.00", "USD"],
    ]
    t = Table(table_data, colWidths=[220, 100, 100])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2C3E50")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.whitesmoke),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ]))
    story.append(t)

    doc.build(story)
    print(f"[OK] Generated test PDF: {filename}")

generate_test_pdf("sample_doc.pdf")

In [ ]:
# SELECT YOUR LLM HERE:
if groq_api_key:
    ACTIVE_LLM = groq_config
    print("[INFO] Using Groq API LLM.")
else:
    # ACTIVE_LLM = MockLLM()
    print("[WARN] No GROQ_API_KEY detected. Using MockLLM for offline verification.")

In [ ]:
import time
from typing import Optional
from pydantic import BaseModel, Field

from docuparser import Docuparser, ParsedDocument, BaseExtension
from docuparser.models.graph import KnowledgeGraph

# Define schema for Table Extraction
class ExpenditureItem(BaseModel):
    item: str = Field(description="Service or item description")
    amount: float = Field(description="Total cost")
    currency: Optional[str] = Field(default="USD")

print("=" * 60)
print("             DOCUPARSER COMPREHENSIVE TEST")
print("=" * 60)

# -------------------------------------------------------------
# Test 1: Parser Initialization & Docling Conversion
# -------------------------------------------------------------
print("\n[TEST 1] Initializing Docuparser & Running Docling Parse...")
t0 = time.time()
parser = Docuparser(
    file_path="colab_test_doc.pdf",
    llm=ACTIVE_LLM,
    do_ocr=True
)
parser.parse()
print(f" -> Document parsed in {time.time() - t0:.2f}s")
print(f" -> Detected Markdown Length: {len(parser.get_markdown())} chars")
assert len(parser.get_markdown()) > 0, "Markdown export failed."
print(" -> [PASS] Test 1 Succeeded.")

In [ ]:
# -------------------------------------------------------------
# Test 2: PII Masking (Presidio + RegEx Fallback)
# -------------------------------------------------------------
print("\n[TEST 2] Testing PII Masking...")
parser.mask_pii()
masked_md = parser.get_markdown()
print(" -> Masked snippet:\n   ", masked_md[:220].replace("\n", " "))
assert "user@example.com" not in masked_md, "Email was not masked!"
assert "123-45-6789" not in masked_md, "SSN was not masked!"
print(" -> [PASS] Test 2 Succeeded.")

In [ ]:
# -------------------------------------------------------------
# Test 3: Structured Table Extraction (Pydantic Schema)
# -------------------------------------------------------------
print("\n[TEST 3] Testing LLM-driven Structured Table Extraction...")
tables = parser.get_clean_tables(ExpenditureItem)
print(f" -> Extracted {len(tables)} table(s)")
assert len(tables) > 0, "No tables extracted."
for t_idx, tbl in enumerate(tables):
    print(f"    Table {t_idx} has {len(tbl)} validated rows:")
    for row in tbl:
        print(f"      - {row.item}: {row.amount} {row.currency}")
        assert isinstance(row, ExpenditureItem)
print(" -> [PASS] Test 3 Succeeded.")

In [ ]:
# -------------------------------------------------------------
# Test 4: SentenceTransformers Document Chunking & Embeddings
# -------------------------------------------------------------
print("\n[TEST 4] Testing Vector Embeddings & Hybrid Chunking...")
try:
    chunks = parser.to_embeddings(model_name="all-MiniLM-L6-v2", chunk_size=512)
    print(f" -> Generated {len(chunks)} chunks.")
    if chunks and chunks[0]["vector"]:
        print(f" -> Embedding Vector Dimension: {len(chunks[0]['vector'])}")
        assert len(chunks[0]["vector"]) == 384, "Unexpected vector dimension!"
    print(" -> [PASS] Test 4 Succeeded.")
except Exception as e:
    print(f" -> [SKIP / FAIL] Embeddings error: {e}")

In [ ]:
# -------------------------------------------------------------
# Test 5: GLiNER Zero-Shot NER & Knowledge Graph Extraction
# -------------------------------------------------------------
print("\n[TEST 5] Testing GLiNER 2 + Knowledge Graph Construction...")
try:
    kg: KnowledgeGraph = parser.get_knowledge_graph(
        labels=["Company", "Person", "Location", "Date"],
        model_name="fastino/gliner2.5-small-v1"
    )
    print(f" -> Extracted {len(kg.nodes)} Entities/Nodes:")
    for n in kg.nodes:
        print(f"    * [{n.label}] {n.name} (id: {n.id})")

    print(f" -> Extracted {len(kg.edges)} Relationships/Edges:")
    for e in kg.edges:
        print(f"    * ({e.source}) --[{e.relation_type}]--> ({e.target})")

    # Verify Cypher Generation
    cypher_queries = kg.to_cypher()
    print(f" -> Generated {len(cypher_queries)} Cypher statements:")
    for q in cypher_queries[:3]:
        print(f"    CYPHER: {q}")

    assert len(kg.nodes) > 0, "No entities detected by GLiNER!"
    print(" -> [PASS] Test 5 Succeeded.")
except Exception as e:
    print(f" -> [SKIP / FAIL] Graph test error: {e}")

In [ ]:
# -------------------------------------------------------------
# Test 6: Custom Extension Pipeline (.pipe())
# -------------------------------------------------------------
print("\n[TEST 6] Testing Open-Closed Extensibility (.pipe())...")

class WordMetricsExtension(BaseExtension):
    """Custom user plugin to compute word and character stats."""
    def run(self, doc: ParsedDocument, llm=None) -> ParsedDocument:
        words = doc.markdown.split()
        doc.artifacts["metrics"] = {
            "word_count": len(words),
            "char_count": len(doc.markdown)
        }
        return doc

# Pipe custom extension into parser
parser.pipe(WordMetricsExtension())
metrics = parser.doc_context.artifacts.get("metrics")
print(" -> Custom Extension Output:", metrics)
assert metrics and metrics["word_count"] > 0
print(" -> [PASS] Test 6 Succeeded.")

In [ ]:
# -------------------------------------------------------------
# Test 7: Executive Summary
# -------------------------------------------------------------
print("\n[TEST 7] Testing Document Summary...")
summary = parser.get_summary()
print(" -> Summary snippet:\n   ", summary[:200].replace("\n", " "), "...\n")
assert len(summary) > 0
print(" -> [PASS] Test 7 Succeeded.")

print("=" * 60)
print("     ALL MODULE TESTS COMPLETED SUCCESSFULLY! ")
print("=" * 60)